<!-- cspell:ignore atleast dalitz lightgray mystnb xscale yscale zorder -->

# Phase space generators

:::{autolink-concat}
:::

Amplitude analysis needs samples of four-momenta that are distributed uniformly over the phase space of a decay. In TensorWaves, such a sample comes from a {class}`.DataGenerator`. There are two families of generators: {class}`.TFPhaseSpaceGenerator` uses [`phasespace`](https://phasespace.readthedocs.io), which can run on NumPy, TensorFlow, or JAX, and {class}`.EvtGenPhaseSpaceGenerator` uses [EvtGen](https://gitlab.cern.ch/evtgen/evtgen) through [`pyevtgen`](https://github.com/redeboer/pyevtgen-test). This page compares their performance and their distributions, so that you can decide which one fits your use case. All of them take the same masses and return a {obj}`.DataSample` of NumPy arrays with the same structure, so they can be swapped without changing the rest of an analysis.

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib_inline.backend_inline import set_matplotlib_formats

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
set_matplotlib_formats("svg")

## Available generators

Despite its name, {class}`.TFPhaseSpaceGenerator` uses NumPy by default and does not require TensorFlow. Its backend is selected with the `backend` argument. Each generator below gets a {class}`.RealNumberGenerator` that it can use directly where possible. The mapping below is the single place that defines which generators this page compares. All tables and figures on this page loop over it.

In [ ]:
from typing import Literal

from tensorwaves.data import (
    EvtGenPhaseSpaceGenerator,
    NumpyUniformRNG,
    TFPhaseSpaceGenerator,
    TFUniformRealNumberGenerator,
)

initial_state_mass = 3.0969
final_state_masses = {0: 0.1350, 1: 0.1396, 2: 0.4937}


def create_phasespace_generator(
    backend: Literal["numpy", "tensorflow", "jax"],
) -> TFPhaseSpaceGenerator:
    generator = TFPhaseSpaceGenerator(
        initial_state_mass, final_state_masses, backend=backend
    )
    generator.show_progress = False
    return generator


generators = {
    "phasespace (NumPy)": (
        create_phasespace_generator("numpy"),
        NumpyUniformRNG(seed=0),
    ),
    "phasespace (TensorFlow)": (
        create_phasespace_generator("tensorflow"),
        TFUniformRealNumberGenerator(seed=0),
    ),
    "phasespace (JAX)": (create_phasespace_generator("jax"), NumpyUniformRNG(seed=0)),
    "EvtGen": (
        EvtGenPhaseSpaceGenerator(initial_state_mass, final_state_masses),
        NumpyUniformRNG(seed=0),
    ),
}

| Generator | Package | Extra | Events | Use of the {class}`.RealNumberGenerator` |
| --- | --- | --- | --- | --- |
| {class}`.TFPhaseSpaceGenerator` with `backend="numpy"` (default) | `phasespace` | `phsp` | unweighted through hit-and-miss | {class}`.NumpyUniformRNG` is used directly, any other type only provides a seed |
| {class}`.TFPhaseSpaceGenerator` with `backend="tensorflow"` | `phasespace` | `phsp,tf` | unweighted through hit-and-miss | {class}`.TFUniformRealNumberGenerator` is used directly, any other type only provides a seed |
| {class}`.TFPhaseSpaceGenerator` with `backend="jax"` | `phasespace` | `phsp,jax` | unweighted through hit-and-miss | any type only provides a seed |
| {class}`.EvtGenPhaseSpaceGenerator` | `pyevtgen` | `evtgen` | unweighted | any type only provides a seed |

{class}`.TFWeightedPhaseSpaceGenerator` takes the same `backend` argument and returns weighted events instead. The installation of these extras is described on {doc}`/install`.

## Generating a sample

Every generator has the same {meth}`~.DataGenerator.generate` method, which returns a {obj}`dict` of final state IDs to arrays of four-momenta $\left(E, p_x, p_y, p_z\right)$ of shape $n \times 4$.

In [ ]:
n_events = 500_000
samples = {
    name: generator.generate(n_events, rng)
    for name, (generator, rng) in generators.items()
}

In [ ]:
pd.DataFrame(
    {
        "Array type": [type(sample["p0"]).__name__ for sample in samples.values()],
        "Final state": [sorted(sample) for sample in samples.values()],
        "Shape": [sample["p0"].shape for sample in samples.values()],
    },
    index=list(samples),
)

## Performance

TensorFlow and JAX compile functions the first time they are called and `phasespace` compiles them again after each change of backend, so we time a separate first call for each generator. After that, we take the fastest of three calls for several sample sizes, which is the common way to limit the influence of other processes on the machine. These timings only give an impression for this three-body decay on the machine that builds this page.

In [ ]:
sample_sizes = [10_000, 100_000, 1_000_000]
n_repeats = 3


def measure(generator, rng, size: int) -> float:
    start = time.perf_counter()
    generator.generate(size, rng)
    return time.perf_counter() - start


first_call = {}
benchmark = {}
for name, (generator, rng) in generators.items():
    first_call[name] = measure(generator, rng, sample_sizes[0])
    benchmark[name] = [
        min(measure(generator, rng, size) for _ in range(n_repeats))
        for size in sample_sizes
    ]

In [ ]:
df_benchmark = pd.DataFrame(benchmark, index=sample_sizes).T
df_benchmark.columns = [f"{size:,} events [s]" for size in sample_sizes]
df_benchmark.insert(0, f"First call, {sample_sizes[0]:,} events [s]", first_call)
df_benchmark.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for name, durations in benchmark.items():
    ax.plot(sample_sizes, durations, marker="o", label=name)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Number of events")
ax.set_ylabel("Time [s]")
ax.legend()
fig.tight_layout()
plt.show()

## Comparing distributions

The events of different generators are not identical, not even with the same seed, but they should follow the same distribution. A uniform phase space sample shows up as a flat Dalitz plot within the kinematic boundary.

In [ ]:
def invariant_mass_squared(p: np.ndarray) -> np.ndarray:
    return p[:, 0] ** 2 - np.sum(p[:, 1:] ** 2, axis=1)


fig, axes = plt.subplots(
    ncols=len(samples), figsize=(3 * len(samples), 3), sharex=True, sharey=True
)
for ax, (name, sample) in zip(np.atleast_1d(axes), samples.items(), strict=True):
    s01 = invariant_mass_squared(sample["p0"] + sample["p1"])
    s12 = invariant_mass_squared(sample["p1"] + sample["p2"])
    ax.hist2d(s01, s12, bins=60, cmin=1, rasterized=True)
    ax.set_title(name, fontsize="small")
    ax.set_xlabel("$m_{01}^2$ [GeV$^2$]")
np.atleast_1d(axes)[0].set_ylabel("$m_{12}^2$ [GeV$^2$]")
fig.tight_layout()
plt.show()

The invariant mass projections should agree between generators. The lower panels show the ratio of each generator to the first generator in the mapping, with the statistical uncertainty of the reference as a band.

In [ ]:
pairs = {"m_{01}": ("p0", "p1"), "m_{02}": ("p0", "p2"), "m_{12}": ("p1", "p2")}
reference = next(iter(samples))
histograms = {}
fig, axes = plt.subplots(
    nrows=2,
    ncols=len(pairs),
    figsize=(4 * len(pairs), 4.5),
    height_ratios=[3, 1],
    sharex="col",
)
for (top, bottom), (label, (a, b)) in zip(axes.T, pairs.items(), strict=True):
    masses = {
        name: np.sqrt(invariant_mass_squared(sample[a] + sample[b]))
        for name, sample in samples.items()
    }
    edges = np.histogram_bin_edges(masses[reference], bins=60)
    counts = {name: np.histogram(m, bins=edges)[0] for name, m in masses.items()}
    histograms[label] = counts
    reference_counts = np.where(counts[reference] > 0, counts[reference], np.nan)
    for name, n in counts.items():
        top.stairs(n, edges, label=name)
        if name != reference:
            bottom.stairs(n / reference_counts, edges, label=name)
    bottom.stairs(
        1 + 1 / np.sqrt(reference_counts),
        edges,
        baseline=1 - 1 / np.sqrt(reference_counts),
        fill=True,
        color="lightgray",
        zorder=0,
    )
    bottom.set_ylim(0.9, 1.1)
    bottom.set_xlabel(f"${label}$ [GeV]")
axes[0, 0].set_ylabel("Events")
axes[1, 0].set_ylabel("Ratio")
axes[0, 0].legend(fontsize="small")
fig.tight_layout()
plt.show()

To quantify the agreement, we compute a two-sample $\chi^2$ per degree of freedom between each generator and the reference, using bins with sufficient statistics. Values around one mean that the samples are statistically compatible.

In [ ]:
def chi2_per_dof(n1: np.ndarray, n2: np.ndarray) -> float:
    mask = n1 + n2 > 50
    return np.sum((n1 - n2)[mask] ** 2 / (n1 + n2)[mask]) / np.count_nonzero(mask)


pd.DataFrame({
    f"${label}$": {
        name: chi2_per_dof(n, counts[reference])
        for name, n in counts.items()
        if name != reference
    }
    for label, counts in histograms.items()
}).round(2)

## Which generator to choose

{class}`.TFPhaseSpaceGenerator` with its default NumPy backend is the default choice and it is what the {doc}`/amplitude-analysis` page uses, because it needs no additional dependencies and no compilation. TensorFlow reproduces samples from earlier TensorWaves versions, when combined with a {class}`.TFUniformRealNumberGenerator`. JAX can be useful if the rest of the analysis already uses it. The {class}`.TFWeightedPhaseSpaceGenerator` variant provides event weights, which is useful for the {class}`.IntensityDistributionGenerator`.

There are two things to keep in mind with these backends:

- The same seed gives a different sample on each backend. Samples are only reproducible within one backend.
- `phasespace` selects its backend per process, and a generator activates its backend each time it generates. Each switch makes `phasespace` recompile, so alternating between backends is slower than the timings above suggest, and generators with different backends should not be used concurrently in threads.

{class}`.EvtGenPhaseSpaceGenerator` works with any {class}`.RealNumberGenerator`, which only provides a seed for the random engine of EvtGen. There are two limitations:

- It provides no weights, because EvtGen applies accept-reject sampling internally.
- Its accept-reject efficiency drops by about a factor of ten for each additional final state particle, which makes it impractically slow beyond about nine particles. A running call cannot be interrupted.